# exp103 Transformer ODE Dual-Head MoE Standalone Exact

기존에는 py파일로 만들어 학습을 시킨 것을 양식에 맞게 노트북 파일로 수정함.
자체적으로 만든 모듈을 임포트하지 않고 exp103 파이프라인을 완전히 구현하는 동시에, 기존 설정의 실행 조건(프로젝트 루트 open/, 3개 시드, 시드당 5-폴드 학습, 대표 베스트 폴드 체크포인트, 저장소 스타일의 시드 앙상블 제출 파일 명명 규칙)을 그대로 만족합니다.


## 핵심 아이디어

exp103은 관측 궤적 마지막 시점의 Frenet 좌표계를 만들고, 최근 8 step의 운동 feature와 정렬 상대 위치를 Transformer에 넣습니다. Transformer가 만든 latent는 RK4 Neural ODE로 진화시키고, `world_aux`, `frenet_aux`, 6-expert MoE fusion head를 함께 학습합니다.

- 입력: `(batch, 8, 12)` Frenet aligned feature
- target: `world_delta 3D + frenet_residual 3D`
- 최종 예측: Frenet residual을 world 좌표로 복원
- loss: Huber + soft R-HIT + multi-threshold hinge + auxiliary + consistency + gate entropy

## 코드 설명 1. 환경 설정

이 셀은 필요한 패키지를 불러오고 exp103 config를 노트북 안에 로드합니다. 실행 결과는 `outputs/notebook/20260529/exp103/` 아래에 저장되며, checkpoint, submission, log 폴더를 미리 만듭니다. GPU가 있으면 CUDA를 사용하고, 없으면 CPU로 실행합니다.

In [1]:

import json, math, os, pickle, random, re, shutil, zipfile
from datetime import datetime
from pathlib import Path

os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
os.environ.setdefault("PYTHONHASHSEED", "0")

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.model_selection import KFold, train_test_split
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, Dataset, TensorDataset

CONFIG = json.loads('{"data_root": "../open", "output_root": "outputs", "run_name": "exp103", "seed": 42, "seeds": [42, 123, 777], "data": {"dataset_type": "trajectory", "feature_type": "position", "scaler_type": "standard", "target_mode": "dual_world_frenet", "frenet_t_win": 8, "frenet_augmentation": true, "aligned_position_features": true, "return_speed": true}, "model_type": "transformer_ode_dual_head_moe", "model": {"input_dim": 12, "d_model": 128, "first_d_model": 48, "nhead": 4, "first_nhead": 4, "num_layers": 4, "first_num_layers": 4, "num_transformers": 2, "ffn_dim": 512, "first_ffn_dim": 192, "dropout": 0.1, "seq_len": 8, "expert_dim": 128, "output_dim": 3, "num_experts": 6, "gate_temperature": 1.0, "use_gate_dynamics": true, "ode_hidden_dim": 256, "ode_steps": 4, "horizon": 2.0, "aux_head_dim": 128, "fusion_hidden_dim": 128}, "split": {"test_size": 0.2, "random_state": 42}, "train": {"batch_size": 256, "epochs": 200, "patience": 50, "log_every": 10, "grad_clip_norm": 2.0}, "optimizer": {"type": "adamw", "params": {"lr": 0.001, "weight_decay": 0.0001}}, "scheduler": {"type": "cosine_annealing_lr", "params": {"T_max": 100}}, "artifacts": {"model_name": "model.pth", "scaler_name": "scaler.pkl", "submission_name": "submission.csv"}, "visualization": {"mode": "misses"}, "metrics": {"r_hit_threshold": 0.01, "error_bin_size": 0.001, "error_bin_max": 0.016}, "postprocess": {"imm_blend": {"enabled": false}}, "research_note": {"enabled": true, "author": "juyoung"}, "loss_type": "dual_world_frenet_loss", "loss": {"r_hit": 0.01, "w_huber": 0.5, "w_rhit": 0.2, "w_hinge": 0.3, "huber_delta": 0.006, "rhit_sigma": 0.003, "hinge_thresholds": [0.006, 0.008, 0.01, 0.012], "hinge_weights": [0.2, 0.3, 0.4, 0.1], "aux_world_weight": 0.1, "aux_frenet_weight": 0.1, "consistency_weight": 0.03, "gate_entropy_weight": 0.001}, "run_date": "20260529", "experiment": {"seq": 103, "label": "Transformer ODE dual-head MoE", "summary": "exp089의 Frenet aligned 입력과 serial Transformer 인코더를 유지하되, pooled latent를 Neural ODE로 진화시킨 뒤 world delta head와 Frenet residual head를 만들고 MoE fusion으로 최종 Frenet residual을 예측하는 실험이다.", "objective": "Transformer가 관측 window의 패턴을 읽고 Neural ODE가 latent dynamics를 보정하며, dual-head와 MoE fusion이 world/Frenet 관점을 함께 활용할 때 exp089 계열 대비 seed 안정성과 near-threshold R-HIT가 개선되는지 확인한다.", "changes": ["data.target_mode=dual_world_frenet을 사용해 target을 [world_delta 3D, frenet_residual 3D] 6차원으로 구성한다.", "입력 feature는 exp089와 같은 Frenet aligned 12차원, t_win=8, temporal augmentation을 유지한다.", "serial Transformer encoder가 pooled latent를 만들고, RK4 Neural ODE가 이 latent를 horizon=2.0 동안 4 step으로 진화시킨다.", "initial latent와 evolved latent를 concat한 shared representation에서 world_aux와 frenet_aux 보조 head를 각각 출력한다.", "shared representation과 두 보조 출력을 6-expert MoE fusion에 넣어 최종 3차원 Frenet residual을 예측한다."], "expected": "Transformer의 표현력과 Neural ODE의 동역학 보정, MoE의 샘플별 분기 효과가 결합되어 exp100~102의 ODE 단독 계열보다 강하고 exp089 대비 다른 seed에서 보완적인 후보가 될 가능성을 본다."}}')


def find_project_root():
    current = Path.cwd().resolve()
    candidates = [current, *current.parents]
    for candidate in candidates:
        if (candidate / "juyoung" / "configs" / "20260529" / "exp103_transformer_ode_dual_head_moe.json").exists():
            return candidate
    return current


def _slug(value):
    slug = re.sub(r"[^0-9A-Za-z]+", "_", str(value).strip())
    slug = re.sub(r"_+", "_", slug).strip("_").lower()
    return slug or "unknown"


def experiment_run_name(config):
    seq = config.get("experiment", {}).get("seq")
    if seq is None:
        return config.get("run_name", "run")
    return f"{int(seq):03d}_{_slug(config.get('model_type', 'model'))}_{_slug(config.get('loss_type', 'loss'))}"


def experiment_submission_name(config):
    seq = config.get("experiment", {}).get("seq")
    if seq is None:
        return config.get("artifacts", {}).get("submission_name", "submission.csv")
    run_date = str(config.get("run_date") or datetime.now().strftime("%Y%m%d"))
    return f"submission_exp_{int(seq):03d}_{run_date}.csv"


def seed_artifact_name(name, seed):
    path = Path(name)
    return f"{path.stem}_{int(seed)}{path.suffix}"


def fold_artifact_name(name, fold):
    path = Path(name)
    return f"{path.stem}_fold{int(fold)}{path.suffix}"


def device_run_group(device):
    return "gpu" if device.type == "cuda" else "cpu"


PROJECT_ROOT = find_project_root()
BASE_DIR = PROJECT_ROOT / "juyoung"
RUN_DATE = str(CONFIG.get("run_date") or datetime.now().strftime("%Y%m%d"))
RUN_NAME = experiment_run_name(CONFIG)
ARTIFACTS = {
    "model_name": "model.pth",
    "scaler_name": "scaler.pkl",
    "submission_name": experiment_submission_name(CONFIG),
}
CONFIG["run_name"] = RUN_NAME
CONFIG["artifacts"] = dict(ARTIFACTS)
OUTPUT_ROOT = BASE_DIR / CONFIG.get("output_root", "outputs")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
RUN_ROOT = OUTPUT_ROOT / "runs" / device_run_group(DEVICE) / RUN_DATE / RUN_NAME
CHECKPOINT_DIR = RUN_ROOT / "checkpoints"
PRED_DIR = RUN_ROOT / "predictions"
LOG_DIR = RUN_ROOT / "logs"
for p in (CHECKPOINT_DIR, PRED_DIR, LOG_DIR):
    p.mkdir(parents=True, exist_ok=True)

print("project_root:", PROJECT_ROOT)
print("device:", DEVICE)
print("run_name:", RUN_NAME)
print("run_root:", RUN_ROOT)
print("submission_name:", ARTIFACTS["submission_name"])


project_root: D:\juyoung\04.dev\fromGit\dacon
device: cuda
run_name: 103_transformer_ode_dual_head_moe_dual_world_frenet_loss
run_root: D:\juyoung\04.dev\fromGit\dacon\juyoung\outputs\runs\gpu\20260529\103_transformer_ode_dual_head_moe_dual_world_frenet_loss
submission_name: submission_exp_103_20260529.csv


## 코드 설명 2. 데이터 위치와 Seed

이 셀은 재현성을 위한 seed 고정 함수와 데이터 탐색 함수를 정의합니다. 노트북 위치가 조금 달라도 `open/` 또는 `open.zip`을 찾아서 사용할 수 있게 했습니다.

In [2]:

def set_seed(seed, deterministic=True):
    random.seed(int(seed))
    np.random.seed(int(seed))
    torch.manual_seed(int(seed))
    if torch.cuda.is_available():
        torch.cuda.manual_seed(int(seed))
        torch.cuda.manual_seed_all(int(seed))
    if deterministic:
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
        torch.use_deterministic_algorithms(True, warn_only=True)


def find_data_root():
    # juyoung/scripts/train.py effectively trains against PROJECT_ROOT/open.
    data_root = PROJECT_ROOT / "open"
    if (data_root / "train").exists() and (data_root / "test").exists() and (data_root / "train_labels.csv").exists():
        return data_root
    zip_path = PROJECT_ROOT / "open.zip"
    if zip_path.exists():
        target = PROJECT_ROOT / "open"
        target.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(zip_path) as fp:
            fp.extractall(target)
        return target
    raise FileNotFoundError(f"Expected data at {data_root} or {zip_path}")


DATA_ROOT = find_data_root()
print("data:", DATA_ROOT)


data: D:\juyoung\04.dev\fromGit\dacon\open


## 코드 설명 3. Frenet Feature와 좌표 복원

이 셀은 exp103의 핵심 feature/target 로직입니다. 마지막 관측 시점의 속도 방향을 tangent로 잡고 normal/binormal을 만들어 Frenet basis를 구성합니다. 입력 feature는 속도, 가속, jerk, 곡률, 회전 cosine, z 방향 비율 등 9개 운동 feature와 Frenet basis로 정렬한 상대 위치 3개를 합친 12차원입니다. 모델이 예측한 Frenet residual은 마지막에 다시 world 좌표로 복원됩니다.

In [3]:
EPS = 1e-8

def frenet_basis_np(d1, d2):
    speed = np.linalg.norm(d1, axis=1, keepdims=True)
    tangent = d1 / (speed + EPS)
    acc = d1 - d2
    acc_parallel = np.sum(acc * tangent, axis=1, keepdims=True) * tangent
    acc_perp = acc - acc_parallel
    normal_norm = np.linalg.norm(acc_perp, axis=1, keepdims=True)
    normal = acc_perp / (normal_norm + EPS)
    fallback = np.zeros_like(normal)
    fallback_axis = np.argmin(np.abs(tangent), axis=1)
    fallback[np.arange(len(tangent)), fallback_axis] = 1.0
    fallback -= np.sum(fallback * tangent, axis=1, keepdims=True) * tangent
    fallback /= np.linalg.norm(fallback, axis=1, keepdims=True) + EPS
    normal = np.where(normal_norm > 1e-5, normal, fallback)
    binormal = np.cross(tangent, normal)
    binormal /= np.linalg.norm(binormal, axis=1, keepdims=True) + EPS
    return tangent.astype(np.float32), normal.astype(np.float32), binormal.astype(np.float32), speed.astype(np.float32)

def window_indices(end_idx, t_win=8):
    idx = list(range(max(3, end_idx - t_win + 1), end_idx + 1))
    return [idx[0]] * (t_win - len(idx)) + idx if len(idx) < t_win else idx

def make_frenet_features(x, end_idx, t_win=8):
    feats = []
    for idx in window_indices(end_idx, t_win):
        d1, d2, d3 = x[:, idx] - x[:, idx-1], x[:, idx-1] - x[:, idx-2], x[:, idx-2] - x[:, idx-3]
        acc, prev_acc = d1 - d2, d2 - d3
        speed = np.linalg.norm(d1, axis=1, keepdims=True)
        prev_speed = np.linalg.norm(d2, axis=1, keepdims=True)
        tangent = d1 / (speed + EPS)
        acc_parallel = np.sum(acc * tangent, axis=1, keepdims=True)
        acc_perp = acc - acc_parallel * tangent
        perp_norm = np.linalg.norm(acc_perp, axis=1, keepdims=True)
        jerk = acc - prev_acc
        jerk_parallel = np.sum(jerk * tangent, axis=1, keepdims=True)
        jerk_perp_norm = np.linalg.norm(jerk - jerk_parallel * tangent, axis=1, keepdims=True)
        turn_cos = np.sum(d1 * d2, axis=1, keepdims=True) / (speed * prev_speed + EPS)
        curvature = perp_norm / (speed + EPS)
        z_fraction = d1[:, 2:3] / (speed + EPS)
        feats.append(np.concatenate([speed, prev_speed/(speed+EPS), acc_parallel/(speed+EPS), perp_norm/(speed+EPS), turn_cos, curvature, jerk_parallel/(speed+EPS), jerk_perp_norm/(speed+EPS), z_fraction], axis=1))
    return np.stack(feats, axis=1).astype(np.float32)

def make_frenet_aligned_position_features(x, end_idx, t_win=8):
    base = make_frenet_features(x, end_idx, t_win)
    idx = window_indices(end_idx, t_win)
    d1, d2 = x[:, end_idx] - x[:, end_idx-1], x[:, end_idx-1] - x[:, end_idx-2]
    tangent, normal, binormal, _ = frenet_basis_np(d1, d2)
    rel = x[:, idx] - x[:, end_idx:end_idx+1]
    aligned = np.stack([np.sum(rel*tangent[:,None,:], axis=2), np.sum(rel*normal[:,None,:], axis=2), np.sum(rel*binormal[:,None,:], axis=2)], axis=2)
    return np.concatenate([base, aligned], axis=2).astype(np.float32)

def make_frenet_target(x, end_idx, y):
    d1, d2 = x[:, end_idx] - x[:, end_idx-1], x[:, end_idx-1] - x[:, end_idx-2]
    tangent, normal, binormal, speed = frenet_basis_np(d1, d2)
    delta, scale = y - x[:, end_idx], speed[:, 0] + EPS
    return np.stack([np.sum(delta*tangent, axis=1)/scale, np.sum(delta*normal, axis=1)/scale, np.sum(delta*binormal, axis=1)/scale], axis=1).astype(np.float32)

def make_frenet_anchor(x, end_idx):
    d1, d2 = x[:, end_idx] - x[:, end_idx-1], x[:, end_idx-1] - x[:, end_idx-2]
    tangent, _, _, speed = frenet_basis_np(d1, d2)
    acc = d1 - d2
    acc_parallel = np.sum(acc * tangent, axis=1)
    acc_perp = acc - acc_parallel[:, None] * tangent
    scale = speed[:, 0] + EPS
    return np.stack([1.98 + 0.96 * acc_parallel / scale, -0.08 * np.linalg.norm(acc_perp, axis=1) / scale, np.zeros(len(x), dtype=np.float32)], axis=1).astype(np.float32)

def make_frenet_residual_target(x, end_idx, y):
    return (make_frenet_target(x, end_idx, y) - make_frenet_anchor(x, end_idx)).astype(np.float32)

def frenet_residual_to_world(x, end_idx, residual):
    coeff = residual + make_frenet_anchor(x, end_idx)
    d1, d2 = x[:, end_idx] - x[:, end_idx-1], x[:, end_idx-1] - x[:, end_idx-2]
    tangent, normal, binormal, speed = frenet_basis_np(d1, d2)
    return (x[:, end_idx] + coeff[:,0:1]*speed*tangent + coeff[:,1:2]*speed*normal + coeff[:,2:3]*speed*binormal).astype(np.float32)

## 코드 설명 4. Dataset, Scaler, KFold

이 셀은 `open/train`, `open/test`, `train_labels.csv`를 읽고 fold별 학습/검증 split을 만듭니다. 원래 exp103 로그와 동일하게 기본 `n_splits=5`의 KFold를 지원합니다. 각 fold에서 train 원본 8000개는 temporal augmentation 후 56000개가 되고, validation은 원본 2000개를 그대로 평가합니다. 학습 target은 `world_delta 3D + frenet_residual 3D`의 6차원 dual target입니다.

In [4]:
def load_positions(path):
    return pd.read_csv(path)[["x", "y", "z"]].to_numpy()

def load_train_raw():
    labels = pd.read_csv(DATA_ROOT / "train_labels.csv").set_index("id")
    files = sorted((DATA_ROOT / "train").glob("TRAIN_*.csv"))
    pos, y, last = [], [], []
    for fp in files:
        arr = load_positions(fp)
        pos.append(arr); y.append(labels.loc[fp.stem, ["x", "y", "z"]].to_numpy(dtype=np.float32)); last.append(arr[-1])
    return np.stack(pos), np.stack(y), np.stack(last), files

def load_test_raw():
    files = sorted((DATA_ROOT / "test").glob("TEST_*.csv"))
    ids, pos, last = [], [], []
    for fp in files:
        arr = load_positions(fp)
        ids.append(fp.stem); pos.append(arr); last.append(arr[-1])
    return ids, np.stack(pos), np.stack(last), files

def build_augmented_x(pos, t_win=8, augment=True):
    seqs, seq_len = [], pos.shape[1]
    if augment:
        for end_idx in range(3, seq_len - 2):
            seqs.append(make_frenet_aligned_position_features(pos, end_idx, t_win))
    seqs.append(make_frenet_aligned_position_features(pos, seq_len - 1, t_win))
    return np.vstack(seqs).astype(np.float32)

def build_dual_targets(pos, y, augment=True):
    targets, seq_len = [], pos.shape[1]
    if augment:
        for end_idx in range(3, seq_len - 2):
            target = pos[:, end_idx + 2]
            targets.append(np.concatenate([target - pos[:, end_idx], make_frenet_residual_target(pos, end_idx, target)], axis=1))
    targets.append(np.concatenate([y - pos[:, -1], make_frenet_residual_target(pos, seq_len - 1, y)], axis=1))
    return np.vstack(targets).astype(np.float32)

def build_speeds(pos, augment=True):
    speeds, seq_len = [], pos.shape[1]
    if augment:
        for end_idx in range(3, seq_len - 2):
            speeds.append(np.linalg.norm(pos[:, end_idx] - pos[:, end_idx-1], axis=1))
    speeds.append(np.linalg.norm(pos[:, -1] - pos[:, -2], axis=1))
    return np.concatenate(speeds).astype(np.float32)

class TrajectoryStandardScaler:
    def __init__(self):
        self.scaler = StandardScaler()
    def fit_transform(self, x):
        n, t, f = x.shape
        return self.scaler.fit_transform(x.reshape(n*t, f)).reshape(n, t, f).astype(np.float32)
    def transform(self, x):
        n, t, f = x.shape
        return self.scaler.transform(x.reshape(n*t, f)).reshape(n, t, f).astype(np.float32)
    def save(self, path):
        path.parent.mkdir(parents=True, exist_ok=True)
        with open(path, "wb") as fp:
            pickle.dump(self, fp)
    @staticmethod
    def load(path):
        with open(path, "rb") as fp:
            return pickle.load(fp)

class TrajectoryDataset(Dataset):
    def __init__(self, x, y, speed=None):
        self.x = torch.from_numpy(x.astype(np.float32)); self.y = torch.from_numpy(y.astype(np.float32))
        self.speed = None if speed is None else torch.from_numpy(speed.astype(np.float32))
    def __len__(self):
        return len(self.x)
    def __getitem__(self, idx):
        return (self.x[idx], self.y[idx]) if self.speed is None else (self.x[idx], self.y[idx], self.speed[idx])

def build_train_val_split(fold_index=None, n_splits=5):
    pos, y, last_pos, train_files = load_train_raw()
    all_indices = np.arange(len(pos))
    if fold_index is None:
        idx_tr, idx_val = train_test_split(
            all_indices,
            test_size=CONFIG["split"]["test_size"],
            random_state=CONFIG["split"]["random_state"],
        )
    else:
        splitter = KFold(n_splits=n_splits, shuffle=True, random_state=CONFIG["split"]["random_state"])
        folds = list(splitter.split(all_indices))
        idx_tr, idx_val = folds[int(fold_index)]

    t_win, augment = CONFIG["data"]["frenet_t_win"], CONFIG["data"]["frenet_augmentation"]
    pos_tr, pos_val = pos[idx_tr], pos[idx_val]
    y_tr_global, y_val_global = y[idx_tr], y[idx_val]
    x_tr, y_tr, speed = build_augmented_x(pos_tr, t_win, augment), build_dual_targets(pos_tr, y_tr_global, augment), build_speeds(pos_tr, augment)
    end_idx = pos_val.shape[1] - 1
    x_val = make_frenet_aligned_position_features(pos_val, end_idx, t_win)
    scaler = TrajectoryStandardScaler()
    x_tr, x_val = scaler.fit_transform(x_tr), scaler.transform(x_val)
    return {
        "train_dataset": TrajectoryDataset(x_tr, y_tr, speed),
        "val_dataset": TrajectoryDataset(x_val, y_val_global),
        "val_targets": y_val_global,
        "val_pos": pos_val,
        "val_end_idx": end_idx,
        "val_indices": idx_val,
        "scaler": scaler,
        "train_files": train_files,
    }

split_preview = build_train_val_split(fold_index=0, n_splits=5)
print("fold preview:", split_preview["train_dataset"][0][0].shape, len(split_preview["train_dataset"]), len(split_preview["val_dataset"]))


fold preview: torch.Size([8, 12]) 56000 2000


## 코드 설명 5. Transformer ODE Dual-Head MoE 모델

이 셀은 exp103 모델 본체입니다. 첫 Transformer는 `first_d_model=48`로 입력을 정리하고, main Transformer는 `d_model=128` latent를 만듭니다. attention pooling으로 얻은 latent를 RK4 Neural ODE가 4 step 진화시키며, world 보조 head, Frenet 보조 head, 6-expert MoE fusion head가 최종 Frenet residual을 예측합니다.

In [5]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, dropout=0.1, seq_len=8):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        t = torch.arange(seq_len, dtype=torch.float32) / max(seq_len - 1, 1)
        div = torch.exp(torch.arange(0, d_model, 2, dtype=torch.float32) * (-math.log(10000.0) / d_model))
        pe = torch.zeros(seq_len, d_model); pe[:, 0::2] = torch.sin(t[:, None] * div); pe[:, 1::2] = torch.cos(t[:, None] * div)
        self.register_buffer("pe", pe[None])
    def forward(self, x):
        return self.dropout(x + self.pe[:, :x.size(1)])

class ODEFunc(nn.Module):
    def __init__(self, hidden_dim, ode_hidden_dim, dropout):
        super().__init__()
        self.net = nn.Sequential(nn.LayerNorm(hidden_dim), nn.Linear(hidden_dim, ode_hidden_dim), nn.Tanh(), nn.Dropout(dropout), nn.Linear(ode_hidden_dim, hidden_dim))
    def forward(self, state):
        return self.net(state)

class FusionExpert(nn.Module):
    def __init__(self, input_dim, hidden_dim, dropout):
        super().__init__()
        self.net = nn.Sequential(nn.LayerNorm(input_dim), nn.Linear(input_dim, hidden_dim), nn.GELU(), nn.Dropout(dropout), nn.Linear(hidden_dim, 3))
    def forward(self, x):
        return self.net(x)

class TrajectoryTransformerODEDualHeadMoE(nn.Module):
    def __init__(self, input_dim=12, d_model=128, first_d_model=48, nhead=4, first_nhead=4, num_layers=4, first_num_layers=4, num_transformers=2, ffn_dim=512, first_ffn_dim=192, dropout=0.1, seq_len=8, expert_dim=128, output_dim=3, num_experts=6, gate_temperature=1.0, use_gate_dynamics=True, ode_hidden_dim=256, ode_steps=4, horizon=2.0, aux_head_dim=128, fusion_hidden_dim=128):
        super().__init__()
        self.use_gate_dynamics, self.gate_temperature, self.ode_steps, self.horizon = bool(use_gate_dynamics), float(gate_temperature), int(ode_steps), float(horizon)
        self.input_proj = nn.Linear(input_dim, first_d_model)
        self.first_pos_enc = PositionalEncoding(first_d_model, dropout, seq_len)
        self.first_encoder = self._enc(first_d_model, first_nhead, first_ffn_dim, dropout, first_num_layers)
        self.first_to_main_proj = nn.Identity() if first_d_model == d_model else nn.Linear(first_d_model, d_model)
        self.main_pos_enc = PositionalEncoding(d_model, dropout, seq_len)
        self.encoders = nn.ModuleList([self._enc(d_model, nhead, ffn_dim, dropout, num_layers) for _ in range(max(num_transformers - 1, 0))])
        self.pool_query = nn.Parameter(torch.randn(1, 1, d_model) * 0.02)
        self.pool_attn = nn.MultiheadAttention(d_model, nhead, dropout=dropout, batch_first=True)
        self.ode_func = ODEFunc(d_model, ode_hidden_dim, dropout)
        shared_dim, fusion_input_dim = d_model * 2, d_model * 2 + output_dim * 2
        self.world_head = self._head(shared_dim, aux_head_dim, dropout, output_dim)
        self.frenet_head = self._head(shared_dim, aux_head_dim, dropout, output_dim)
        self.experts = nn.ModuleList([FusionExpert(fusion_input_dim, fusion_hidden_dim, dropout) for _ in range(num_experts)])
        gate_input_dim = fusion_input_dim + input_dim * 2 if self.use_gate_dynamics else fusion_input_dim
        self.gate = nn.Sequential(nn.LayerNorm(gate_input_dim), nn.Linear(gate_input_dim, expert_dim), nn.GELU(), nn.Dropout(dropout), nn.Linear(expert_dim, num_experts))
        nn.init.zeros_(self.gate[-1].weight); nn.init.zeros_(self.gate[-1].bias)
    @staticmethod
    def _enc(d_model, nhead, ffn_dim, dropout, num_layers):
        layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead, dim_feedforward=ffn_dim, dropout=dropout, batch_first=True, norm_first=True)
        return nn.TransformerEncoder(layer, num_layers=num_layers)
    @staticmethod
    def _head(input_dim, hidden_dim, dropout, output_dim):
        return nn.Sequential(nn.LayerNorm(input_dim), nn.Linear(input_dim, hidden_dim), nn.GELU(), nn.Dropout(dropout), nn.Linear(hidden_dim, output_dim))
    def integrate(self, state):
        dt, evolved = self.horizon / float(self.ode_steps), state
        for _ in range(self.ode_steps):
            k1 = self.ode_func(evolved); k2 = self.ode_func(evolved + 0.5*dt*k1); k3 = self.ode_func(evolved + 0.5*dt*k2); k4 = self.ode_func(evolved + dt*k3)
            evolved = evolved + (dt / 6.0) * (k1 + 2*k2 + 2*k3 + k4)
        return evolved
    def _encode(self, x):
        h = self.first_encoder(self.first_pos_enc(self.input_proj(x)))
        h = self.main_pos_enc(self.first_to_main_proj(h))
        for enc in self.encoders:
            h = enc(h)
        q = self.pool_query.expand(x.size(0), -1, -1)
        return self.pool_attn(q, h, h)[0].squeeze(1)
    def return_details(self, x):
        initial = self._encode(x); evolved = self.integrate(initial); shared = torch.cat([initial, evolved], dim=1)
        world_aux, frenet_aux = self.world_head(shared), self.frenet_head(shared)
        fusion_input = torch.cat([shared, world_aux, frenet_aux], dim=1)
        if self.use_gate_dynamics:
            last_step, prev_step = x[:, -1] - x[:, -2], x[:, -2] - x[:, -3]
            gate_input = torch.cat([fusion_input, last_step, last_step - prev_step], dim=1)
        else:
            gate_input = fusion_input
        weights = torch.softmax(self.gate(gate_input) / self.gate_temperature, dim=1)
        experts = torch.stack([expert(fusion_input) for expert in self.experts], dim=1)
        return {"prediction": torch.sum(weights[:, :, None] * experts, dim=1), "world_aux": world_aux, "frenet_aux": frenet_aux, "weights": weights, "experts": experts}
    def forward(self, x):
        return self.return_details(x)["prediction"]

model_preview = TrajectoryTransformerODEDualHeadMoE(**CONFIG["model"]).to(DEVICE)
with torch.no_grad():
    details = model_preview.return_details(split_preview["train_dataset"][0][0][None].to(DEVICE))
print(details["prediction"].shape, details["weights"].shape)
del model_preview

C:\Users\고래들\AppData\Local\Temp\ipykernel_84452\2325957807.py:49: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  return nn.TransformerEncoder(layer, num_layers=num_layers)


torch.Size([1, 3]) torch.Size([1, 6])


## 코드 설명 6. Loss와 Metric

이 셀은 dual target에 맞춘 손실 함수입니다. 메인 Frenet residual에는 Huber, soft R-HIT, multi-threshold hinge를 섞고, world auxiliary loss와 Frenet auxiliary loss를 추가합니다. consistency loss는 world head의 delta 크기와 Frenet head를 world 거리로 환산한 크기가 너무 어긋나지 않게 합니다.

In [6]:
class FrenetWorldMoELoss(nn.Module):
    def __init__(self, r_hit=0.01, w_huber=0.5, w_rhit=0.2, w_hinge=0.3, huber_delta=0.006, rhit_sigma=0.003, hinge_thresholds=(0.006,0.008,0.010,0.012), hinge_weights=(0.2,0.3,0.4,0.1)):
        super().__init__()
        self.r_hit, self.w_huber, self.w_rhit, self.w_hinge = float(r_hit), float(w_huber), float(w_rhit), float(w_hinge)
        self.huber_delta, self.rhit_sigma = float(huber_delta), float(rhit_sigma)
        self.hinge_thresholds, self.hinge_weights = tuple(map(float, hinge_thresholds)), tuple(map(float, hinge_weights))
    def forward(self, y_pred, y_true, speed):
        frenet_dist = torch.norm(y_pred - y_true, dim=1)
        world_dist = speed.to(frenet_dist.dtype).clamp_min(1e-8) * frenet_dist
        huber = F.huber_loss(frenet_dist, torch.zeros_like(frenet_dist), delta=self.huber_delta)
        rhit = 1.0 - torch.sigmoid((self.r_hit - world_dist) / self.rhit_sigma).mean()
        hinge = y_pred.new_tensor(0.0)
        for th, w in zip(self.hinge_thresholds, self.hinge_weights):
            hinge = hinge + w * torch.clamp(world_dist - th, min=0.0).mean()
        return self.w_huber*huber + self.w_rhit*rhit + self.w_hinge*hinge/max(sum(self.hinge_weights), 1e-8)

class DualWorldFrenetLoss(nn.Module):
    def __init__(self, aux_world_weight=0.1, aux_frenet_weight=0.1, consistency_weight=0.03, gate_entropy_weight=0.001, **kwargs):
        super().__init__()
        self.frenet_loss = FrenetWorldMoELoss(**kwargs)
        self.r_hit, self.rhit_sigma = kwargs.get("r_hit", 0.01), kwargs.get("rhit_sigma", 0.003)
        self.aux_world_weight, self.aux_frenet_weight = aux_world_weight, aux_frenet_weight
        self.consistency_weight, self.gate_entropy_weight = consistency_weight, gate_entropy_weight
    def _world_delta_loss(self, pred, target):
        dist = torch.norm(pred - target, dim=1)
        huber = F.huber_loss(dist, torch.zeros_like(dist), delta=0.006)
        rhit = 1.0 - torch.sigmoid((self.r_hit - dist) / self.rhit_sigma).mean()
        return 0.7*huber + 0.3*rhit
    def forward(self, details, y_true, speed):
        target_world, target_frenet = y_true[:, :3], y_true[:, 3:6]
        loss = self.frenet_loss(details["prediction"], target_frenet, speed)
        loss = loss + self.aux_world_weight * self._world_delta_loss(details["world_aux"], target_world)
        loss = loss + self.aux_frenet_weight * self.frenet_loss(details["frenet_aux"], target_frenet, speed)
        world_norm = torch.norm(details["world_aux"], dim=1)
        frenet_world_norm = speed.to(world_norm.dtype).clamp_min(1e-8) * torch.norm(details["frenet_aux"], dim=1)
        loss = loss + self.consistency_weight * F.smooth_l1_loss(world_norm, frenet_world_norm)
        weights = details["weights"].clamp_min(1e-8)
        return loss - self.gate_entropy_weight * (-(weights * torch.log(weights)).sum(dim=1).mean())

def r_hit_score(pred, true, threshold=0.01):
    return float((np.linalg.norm(pred - true, axis=1) <= threshold).mean())

def error_bin_counts(errors, bin_size=0.001, max_error=0.016):
    counts, edge = {}, 0.0
    while edge < max_error - 1e-12:
        nxt = edge + bin_size
        counts[f"{edge:.3f}-{nxt:.3f}"] = int(((errors >= edge) & (errors < nxt)).sum())
        edge = nxt
    counts[f">={max_error:.3f}"] = int((errors >= max_error).sum())
    return counts

## 코드 설명 7. Seed별 5-Fold 학습 루프

이 셀은 원래 exp103 실행 로그처럼 seed 하나마다 5개 fold를 학습합니다. 각 fold는 별도 model/scaler를 저장하고, epoch마다 active fold의 평균 R-HIT를 기록합니다. 마지막에는 가장 best R-HIT가 높은 fold를 대표 checkpoint(`model.pth`, `model_123.pth`, `model_777.pth`)로 복사하고 `kfold_summary*.json`을 저장합니다.

In [7]:

def predict_absolute(model, loader, split):
    model.eval()
    preds = []
    with torch.no_grad():
        for batch in loader:
            preds.append(model(batch[0].to(DEVICE)).cpu().numpy())
    return frenet_residual_to_world(split["val_pos"], split["val_end_idx"], np.concatenate(preds))


def artifact_names_for_seed(seed, seed_index):
    artifacts = dict(ARTIFACTS)
    if seed_index > 0:
        artifacts = {key: seed_artifact_name(value, seed) for key, value in artifacts.items()}
    return artifacts


def train_one_seed(seed, seed_index=0, seed_count=1, n_splits=5):
    seed = int(seed)
    set_seed(seed)
    artifacts = artifact_names_for_seed(seed, seed_index)

    # Match juyoung/scripts/train.py exactly: _run_train_single builds the normal
    # train/val split and initializes one model before dispatching to kfold. That
    # representative model is not used for kfold, but it consumes torch RNG and
    # therefore changes every fold model initialization if omitted.
    _pre_split = build_train_val_split(fold_index=None, n_splits=n_splits)
    _pre_generator = torch.Generator().manual_seed(seed)
    _pre_train_loader = DataLoader(_pre_split["train_dataset"], batch_size=CONFIG["train"]["batch_size"], shuffle=True, generator=_pre_generator)
    _pre_val_loader = DataLoader(_pre_split["val_dataset"], batch_size=CONFIG["train"]["batch_size"], shuffle=False, generator=_pre_generator)
    _pre_model = TrajectoryTransformerODEDualHeadMoE(**CONFIG["model"]).to(DEVICE)
    _pre_optimizer = torch.optim.AdamW(_pre_model.parameters(), **CONFIG["optimizer"]["params"])
    _pre_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(_pre_optimizer, **CONFIG["scheduler"]["params"])
    _pre_criterion = DualWorldFrenetLoss(**CONFIG["loss"])
    del _pre_split, _pre_generator, _pre_train_loader, _pre_val_loader, _pre_model, _pre_optimizer, _pre_scheduler, _pre_criterion

    fold_states = []
    print(f"Start {n_splits}-fold training for seed={seed} ({seed_index + 1}/{seed_count})")

    for fold_index in range(n_splits):
        split = build_train_val_split(fold_index=fold_index, n_splits=n_splits)
        generator = torch.Generator().manual_seed(seed + fold_index)
        train_loader = DataLoader(split["train_dataset"], batch_size=CONFIG["train"]["batch_size"], shuffle=True, generator=generator)
        val_loader = DataLoader(split["val_dataset"], batch_size=CONFIG["train"]["batch_size"], shuffle=False, generator=generator)
        model = TrajectoryTransformerODEDualHeadMoE(**CONFIG["model"]).to(DEVICE)
        optimizer = torch.optim.AdamW(model.parameters(), **CONFIG["optimizer"]["params"])
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, **CONFIG["scheduler"]["params"])
        fold = fold_index + 1
        fold_states.append({
            "fold": fold,
            "split": split,
            "train_loader": train_loader,
            "val_loader": val_loader,
            "model": model,
            "optimizer": optimizer,
            "scheduler": scheduler,
            "criterion": DualWorldFrenetLoss(**CONFIG["loss"]),
            "best_hit": -1.0,
            "patience_count": 0,
            "stopped": False,
            "model_path": CHECKPOINT_DIR / fold_artifact_name(artifacts["model_name"], fold),
            "scaler_path": CHECKPOINT_DIR / fold_artifact_name(artifacts["scaler_name"], fold),
        })
        print(f"Fold {fold}/{n_splits}: train={len(split['train_dataset'])} val={len(split['val_dataset'])} shape={split['train_dataset'][0][0].shape}")

    best_avg_hit = -1.0
    rows = []
    for epoch in range(1, CONFIG["train"]["epochs"] + 1):
        epoch_losses, epoch_hits, epoch_mean_errors, active_count = [], [], [], 0
        for state in fold_states:
            if state["stopped"]:
                continue
            active_count += 1
            model = state["model"]
            model.train()
            train_loss = 0.0
            for x, y, speed in state["train_loader"]:
                x = x.to(DEVICE)
                y = y.to(DEVICE).to(torch.float32)
                speed = speed.to(DEVICE).to(torch.float32)
                state["optimizer"].zero_grad()
                loss = state["criterion"](model.return_details(x), y, speed)
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), float(CONFIG["train"]["grad_clip_norm"]))
                state["optimizer"].step()
                train_loss += loss.item() * len(x)
            if state["scheduler"] is not None:
                state["scheduler"].step()
            train_loss /= len(state["split"]["train_dataset"])

            val_pred = predict_absolute(model, state["val_loader"], state["split"])
            errors = np.linalg.norm(val_pred - state["split"]["val_targets"], axis=1)
            val_hit = r_hit_score(val_pred, state["split"]["val_targets"], CONFIG["metrics"]["r_hit_threshold"])
            epoch_losses.append(train_loss)
            epoch_hits.append(val_hit)
            epoch_mean_errors.append(float(errors.mean()))

            if val_hit > state["best_hit"]:
                state["best_hit"] = val_hit
                state["patience_count"] = 0
                torch.save({"model_state": model.state_dict(), "config": CONFIG}, state["model_path"])
                state["split"]["scaler"].save(state["scaler_path"])
                print(f"Fold {state['fold']} best updated at epoch {epoch} with val R-HIT={val_hit:.4f}")
            else:
                state["patience_count"] += 1
                if state["patience_count"] >= CONFIG["train"]["patience"]:
                    state["stopped"] = True
                    print(f"Fold {state['fold']} early stopping epoch {epoch} (best={state['best_hit']:.4f})")

        if not epoch_hits:
            print(f"All folds stopped before epoch {epoch}.")
            break

        avg_loss = float(np.mean(epoch_losses))
        avg_hit = float(np.mean(epoch_hits))
        avg_mean_error = float(np.mean(epoch_mean_errors))
        rows.append({"epoch": epoch, "train_loss": avg_loss, "r_hit": avg_hit, "mean_error": avg_mean_error, "active_folds": active_count})
        best_avg_hit = max(best_avg_hit, avg_hit)
        if epoch % CONFIG["train"]["log_every"] == 0:
            print(f"Epoch {epoch:3d}/{CONFIG['train']['epochs']} folds={active_count} avg_loss={avg_loss:.6f} avg_val_R-HIT={avg_hit:.4f} avg_mean_error={avg_mean_error:.5f}")

    best_state = max(fold_states, key=lambda state: state["best_hit"])
    final_model_path = CHECKPOINT_DIR / artifacts["model_name"]
    final_scaler_path = CHECKPOINT_DIR / artifacts["scaler_name"]
    shutil.copy2(best_state["model_path"], final_model_path)
    shutil.copy2(best_state["scaler_path"], final_scaler_path)

    pd.DataFrame(rows).to_csv(LOG_DIR / ("history.csv" if seed_index == 0 else f"history_{seed}.csv"), index=False)
    fold_summary = [{"fold": int(state["fold"]), "best_hit": float(state["best_hit"]), "model_path": str(state["model_path"]), "scaler_path": str(state["scaler_path"])} for state in fold_states]
    (RUN_ROOT / "kfold_summary.json").write_text(json.dumps({"n_splits": n_splits, "best_avg_epoch_r_hit": float(best_avg_hit), "folds": fold_summary, "selected_fold": int(best_state["fold"])}, ensure_ascii=False, indent=2), encoding="utf-8")
    return {
        "seed": seed,
        "score": float(best_avg_hit),
        "score_name": "validation_r_hit",
        "best_hit": float(best_avg_hit),
        "device": str(DEVICE),
        "device_type": DEVICE.type,
        "run_group": device_run_group(DEVICE),
        "model_path": str(final_model_path),
        "scaler_path": str(final_scaler_path),
        "submission_model_path": None,
        "submission_scaler_path": None,
        "log_path": str(LOG_DIR / ("train.log" if seed_index == 0 else f"train_{seed}.log")),
        "visualization_saved": seed_index == 0 and CONFIG.get("visualization", {}).get("mode", "none") != "none",
    }


seed_results = [train_one_seed(seed, seed_index=index, seed_count=len(CONFIG["seeds"]), n_splits=5) for index, seed in enumerate(CONFIG["seeds"])]
(RUN_ROOT / "seed_summary.json").write_text(json.dumps(seed_results, ensure_ascii=False, indent=2), encoding="utf-8")
pd.DataFrame(seed_results).to_csv(LOG_DIR / "seed_summary.csv", index=False)
seed_results


C:\Users\고래들\AppData\Local\Temp\ipykernel_84452\2325957807.py:49: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  return nn.TransformerEncoder(layer, num_layers=num_layers)


Start 5-fold training for seed=42 (1/3)


C:\Users\고래들\AppData\Local\Temp\ipykernel_84452\2325957807.py:49: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  return nn.TransformerEncoder(layer, num_layers=num_layers)


Fold 1/5: train=56000 val=2000 shape=torch.Size([8, 12])


C:\Users\고래들\AppData\Local\Temp\ipykernel_84452\2325957807.py:49: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  return nn.TransformerEncoder(layer, num_layers=num_layers)


...

C:\Users\고래들\AppData\Local\Temp\ipykernel_84452\2325957807.py:49: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  return nn.TransformerEncoder(layer, num_layers=num_layers)


Fold 4/5: train=56000 val=2000 shape=torch.Size([8, 12])


C:\Users\고래들\AppData\Local\Temp\ipykernel_84452\2325957807.py:49: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  return nn.TransformerEncoder(layer, num_layers=num_layers)


Fold 5/5: train=56000 val=2000 shape=torch.Size([8, 12])
Fold 1 best updated at epoch 1 with val R-HIT=0.6550
Fold 2 best updated at epoch 1 with val R-HIT=0.6530
Fold 3 best updated at epoch 1 with val R-HIT=0.6575
Fold 4 best updated at epoch 1 with val R-HIT=0.6395
Fold 5 best updated at epoch 1 with val R-HIT=0.6505
Fold 4 best updated at epoch 2 with val R-HIT=0.6420
Fold 5 best updated at epoch 2 with val R-HIT=0.6590
Fold 2 best updated at epoch 3 with val R-HIT=0.6610
Fold 4 best updated at epoch 3 with val R-HIT=0.6470
Fold 5 best updated at epoch 3 with val R-HIT=0.6610
Fold 1 best updated at epoch 4 with val R-HIT=0.6570
Fold 2 best updated at epoch 4 with val R-HIT=0.6640
Fold 4 best updated at epoch 4 with val R-HIT=0.6490
Fold 1 best updated at epoch 5 with val R-HIT=0.6660
Fold 4 best updated at epoch 5 with val R-HIT=0.6500
Fold 5 best updated at epoch 5 with val R-HIT=0.6625
Fold 2 best updated at epoch 6 with val R-HIT=0.6645
Fold 3 best updated at epoch 6 with val R-

[{'seed': 42,
  'score': 0.6825,
  'score_name': 'validation_r_hit',
  'best_hit': 0.6825,
  'device': 'cuda',
  'device_type': 'cuda',
  'run_group': 'gpu',
  'model_path': 'D:\\juyoung\\04.dev\\fromGit\\dacon\\juyoung\\outputs\\runs\\gpu\\20260529\\103_transformer_ode_dual_head_moe_dual_world_frenet_loss\\checkpoints\\model.pth',
  'scaler_path': 'D:\\juyoung\\04.dev\\fromGit\\dacon\\juyoung\\outputs\\runs\\gpu\\20260529\\103_transformer_ode_dual_head_moe_dual_world_frenet_loss\\checkpoints\\scaler.pkl',
  'submission_model_path': None,
  'submission_scaler_path': None,
  'log_path': 'D:\\juyoung\\04.dev\\fromGit\\dacon\\juyoung\\outputs\\runs\\gpu\\20260529\\103_transformer_ode_dual_head_moe_dual_world_frenet_loss\\logs\\train.log',
  'visualization_saved': True},
 {'seed': 123,
  'score': 0.687,
  'score_name': 'validation_r_hit',
  'best_hit': 0.687,
  'device': 'cuda',
  'device_type': 'cuda',
  'run_group': 'gpu',
  'model_path': 'D:\\juyoung\\04.dev\\fromGit\\dacon\\juyoung\\ou

## 코드 설명 8. 추론과 Seed Ensemble

이 셀은 KFold 학습 후 선택된 대표 fold checkpoint와 scaler를 읽어 test feature를 만들고 submission을 저장합니다. seed 42는 `submission.csv`, seed 123/777은 `submission_123.csv`, `submission_777.csv`처럼 저장되며, 필요하면 seed 평균 제출인 `submission_seed_mean.csv`도 만들 수 있습니다.

In [10]:

def build_test_bundle(scaler):
    ids, pos, last_pos, files = load_test_raw()
    end_idx = pos.shape[1] - 1
    x = make_frenet_aligned_position_features(pos, end_idx, CONFIG["data"]["frenet_t_win"])
    return {"ids": ids, "inputs": scaler.transform(x), "last_pos": last_pos, "pos": pos, "end_idx": end_idx, "files": files}


def submission_name_with_metadata(name, device=None, score=None):
    if device is None and score is None:
        return name
    path = Path(name)
    parts = [path.stem]
    if device:
        parts.append(str(device).replace(" ", "_").replace(":", "_"))
    if score is not None:
        parts.append(f"{float(score):.4f}")
    return "_".join(parts) + path.suffix


def save_submission(ids, preds, output_path):
    sample = pd.read_csv(DATA_ROOT / "sample_submission.csv")
    pred_df = pd.DataFrame({"id": ids, "x": preds[:, 0], "y": preds[:, 1], "z": preds[:, 2]})
    sample[["id"]].merge(pred_df, on="id", how="left").to_csv(output_path, index=False)
    print("saved", output_path)
    return output_path


def inference_one_seed(seed, seed_index=0, seed_count=1):
    seed = int(seed)
    set_seed(seed)
    artifacts = artifact_names_for_seed(seed, seed_index)
    model_path = CHECKPOINT_DIR / artifacts["model_name"]
    scaler_path = CHECKPOINT_DIR / artifacts["scaler_name"]
    score = seed_results[seed_index].get("score") if "seed_results" in globals() else None
    submission_name = submission_name_with_metadata(artifacts["submission_name"], device=device_run_group(DEVICE), score=score)
    submission_path = PRED_DIR / submission_name

    scaler = TrajectoryStandardScaler.load(scaler_path)
    test = build_test_bundle(scaler)
    model = TrajectoryTransformerODEDualHeadMoE(**CONFIG["model"]).to(DEVICE)
    checkpoint = torch.load(model_path, map_location=DEVICE)
    model.load_state_dict(checkpoint["model_state"])
    model.eval()
    loader = DataLoader(TensorDataset(torch.from_numpy(test["inputs"]), torch.zeros((len(test["inputs"]), 3), dtype=torch.float32)), batch_size=CONFIG["train"]["batch_size"], shuffle=False)
    raw = []
    with torch.no_grad():
        for batch in loader:
            raw.append(model(batch[0].to(DEVICE)).cpu().numpy())
    preds = frenet_residual_to_world(test["pos"], test["end_idx"], np.concatenate(raw))
    save_submission(test["ids"], preds, submission_path)
    return {"seed": seed, "score": score, "score_name": "validation_r_hit", "best_hit": score, "device": str(DEVICE), "device_type": DEVICE.type, "run_group": device_run_group(DEVICE), "run_root": str(RUN_ROOT), "checkpoint_root": str(RUN_ROOT), "model_path": str(model_path), "scaler_path": str(scaler_path), "submission_path": str(submission_path)}


def make_mean_submission(results, output_path=PRED_DIR / ARTIFACTS["submission_name"]):
    frames = [pd.read_csv(result["submission_path"]) for result in results]
    ids = frames[0]["id"].tolist()
    for frame in frames[1:]:
        if frame["id"].tolist() != ids:
            raise ValueError("submission id order mismatch")
    out = frames[0][["id"]].copy()
    out[["x", "y", "z"]] = np.mean([frame[["x", "y", "z"]].to_numpy(float) for frame in frames], axis=0)
    out.to_csv(output_path, index=False)
    print("saved", output_path)
    return output_path


inference_results = [inference_one_seed(seed, seed_index=index, seed_count=len(CONFIG["seeds"])) for index, seed in enumerate(CONFIG["seeds"])]
(PRED_DIR / "submission_summary.json").write_text(json.dumps(inference_results, ensure_ascii=False, indent=2), encoding="utf-8")
pd.DataFrame(inference_results).to_csv(PRED_DIR / "submission_summary.csv", index=False)
make_mean_submission(inference_results)


C:\Users\고래들\AppData\Local\Temp\ipykernel_84452\2325957807.py:49: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  return nn.TransformerEncoder(layer, num_layers=num_layers)


saved D:\juyoung\04.dev\fromGit\dacon\juyoung\outputs\runs\gpu\20260529\103_transformer_ode_dual_head_moe_dual_world_frenet_loss\predictions\submission_exp_103_20260529_gpu_0.6825.csv


C:\Users\고래들\AppData\Local\Temp\ipykernel_84452\2325957807.py:49: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  return nn.TransformerEncoder(layer, num_layers=num_layers)


saved D:\juyoung\04.dev\fromGit\dacon\juyoung\outputs\runs\gpu\20260529\103_transformer_ode_dual_head_moe_dual_world_frenet_loss\predictions\submission_exp_103_20260529_123_gpu_0.6870.csv


C:\Users\고래들\AppData\Local\Temp\ipykernel_84452\2325957807.py:49: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  return nn.TransformerEncoder(layer, num_layers=num_layers)


saved D:\juyoung\04.dev\fromGit\dacon\juyoung\outputs\runs\gpu\20260529\103_transformer_ode_dual_head_moe_dual_world_frenet_loss\predictions\submission_exp_103_20260529_777_gpu_0.6830.csv
saved D:\juyoung\04.dev\fromGit\dacon\juyoung\outputs\runs\gpu\20260529\103_transformer_ode_dual_head_moe_dual_world_frenet_loss\predictions\submission_exp_103_20260529.csv


WindowsPath('D:/juyoung/04.dev/fromGit/dacon/juyoung/outputs/runs/gpu/20260529/103_transformer_ode_dual_head_moe_dual_world_frenet_loss/predictions/submission_exp_103_20260529.csv')